# 小米 14 初版 → 最新版：完整性复核

## tl;dr

本场在第一次翻译之前因频率门槛超时终止，成功进程为 0/48。不能计算速度、内存或质量差值。

## Context & Methods

### Key Assumptions

- 同一台小米 14，v0.1.0 与本地 v0.3.0，共用测试入口、模型及语料。
- 固定八场景 × 两版本 × 三轮，每进程三遍；只从本场原始文件复核。
- 缺失值不是零；不拼接其他场次，不降低频率或稳定性门槛。
- 构建成功不代表测试完成；频率门槛失败也不证明引擎性能回归。

## Data

原始文件为 `attempt-1.json`；采集器与测试入口快照位于 `collector/`。构建及输入摘要在原始报告内。

In [1]:
from pathlib import Path
import json
import runpy
from IPython.display import Markdown, display
root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'tools/version-bench/check.py').is_file())
record = root / 'benchmarks/v0.3.0/mi14-2026-09-09/initial-to-current'
checks = runpy.run_path(str(record / 'verify.py'))
result = checks['verify']('attempt-1.json')
raw = json.loads((record / 'attempt-1.json').read_text())

## Results

### 1. 验证缺失状态与原始退出原因

In [2]:
assert result['successful_processes'] == 0
assert result['planned_processes'] == 48
assert not result['accepted'] and result['exit_code'] == 2
assert len(result['rows']) == 16
assert all(not row['complete'] and 'cold_inputs_per_second' not in row for row in result['rows'])
assert len(raw['runs']) == 1 and raw['runs'][0]['samples'] == []
print({k: v for k, v in result.items() if k != 'rows'})
print(json.dumps(raw['runs'][0], ensure_ascii=False, indent=2))

{'file': 'attempt-1.json', 'accepted': False, 'exit_code': 2, 'successful_processes': 0, 'planned_processes': 48, 'diagnostic': 'INVALID: v0.1.0: missing or unexpected scenarios'}
{
  "round": 1,
  "scenario": "enzh_w1",
  "version": "v0.1.0",
  "before": {
    "max_freq_khz": {
      "2": "2630400",
      "7": "2169600"
    },
    "battery_temp_c": 32.5,
    "board_temp": "33498",
    "foreground": {
      "wakefulness": "Awake",
      "lockscreen": false,
      "focused_package": "io.github.yinvoker.bergamot.bench"
    },
    "gate_wait_s": -1
  },
  "returncode": 2,
  "samples": [],
  "error": "frequency gate timeout; not measured"
}


### 2. 保留所有场景，不生成缺失性能的中位数

In [3]:
rows = ['| 场景 | 版本 | 成功进程 | 状态 |', '|---|---|---:|---|']
for row in result['rows']:
    status = '; '.join(row['errors']) or '未测'
    rows.append(f"| {row['scenario']} | {row['version']} | {row['successful_processes']}/3 | {status} |")
display(Markdown('\n'.join(rows)))

| 场景 | 版本 | 成功进程 | 状态 |
|---|---|---:|---|
| enzh_w1 | v0.1.0 | 0/3 | frequency gate timeout; not measured |
| enzh_w1 | v0.3.0 | 0/3 | 未测 |
| enzh_w2 | v0.1.0 | 0/3 | 未测 |
| enzh_w2 | v0.3.0 | 0/3 | 未测 |
| enzh_w4 | v0.1.0 | 0/3 | 未测 |
| enzh_w4 | v0.3.0 | 0/3 | 未测 |
| pivot_w1 | v0.1.0 | 0/3 | 未测 |
| pivot_w1 | v0.3.0 | 0/3 | 未测 |
| enzh_b512p | v0.1.0 | 0/3 | 未测 |
| enzh_b512p | v0.3.0 | 0/3 | 未测 |
| pivot_b512p | v0.1.0 | 0/3 | 未测 |
| pivot_b512p | v0.3.0 | 0/3 | 未测 |
| enzh_w2_512p | v0.1.0 | 0/3 | 未测 |
| enzh_w2_512p | v0.3.0 | 0/3 | 未测 |
| pivot_w2_512p | v0.1.0 | 0/3 | 未测 |
| pivot_w2_512p | v0.3.0 | 0/3 | 未测 |

## Takeaways

本场不支持任何版本间性能结论。复跑须使用新的完整场次，保留此失败记录。项目 README 中的小米 10 历史对照暂不改为小米 14；同日 v0.2.0 → v0.3.0 的通过记录也不能代替缺失的 v0.1.0 数据。